# Multiscale TopoReg — Graph Transformer Encoder

This notebook is fully self-contained. It replaces the original GCN/GAT encoder with a **Graph Transformer** encoder while keeping every other part of the system identical — topology-preserving loss, community graph construction, training loop, and evaluation are unchanged.

**Encoder:** `GraphTransformerConv` — full multi-head self-attention (4 heads by default) with a Graphormer-style structural bias: `attn[i,j,h] += adj[i,j] × edge_bias[h]`. This unifies the Graphormer (spatial/structural encoding) and SAN (global attention + local structure) design principles in a single layer. Memory is O(N² × H); suitable for the node sizes in these datasets.

Saves to `Output/Encoder Variants/multiscale_toporeg_transformer` to keep results separate from other variants.

In [1]:
!pip install gudhi pot -q
!pip install networkx
!pip install pathlib
!pip install dataclasses
!pip install typing
!pip install numpy==1.23.5
!pip install scikit-learn
!pip install scipy
!pip install torch==1.12.1+cu113 torchvision==0.13.1+cu113 torchaudio==0.12.1 -f https://download.pytorch.org/whl/torch_stable.html
!pip install eagerpy

Looking in links: https://download.pytorch.org/whl/torch_stable.html


In [2]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import json
import pickle
import random

import gudhi as gd
from gudhi.wasserstein import wasserstein_distance
import networkx as nx
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn import metrics
from sklearn.cluster import KMeans

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

graph_pkl = ["enron", "highschool", "DBLP", "Cora", "DBLPdyn"]
label_num_dic = {"Cora": 10, "enron": 7, "highschool": 9, "DBLP": 15, "DBLPdyn": 14}


def glorot_init(input_dim: int, output_dim: int) -> nn.Parameter:
    init_range = np.sqrt(6.0 / (input_dim + output_dim))
    initial = torch.rand(input_dim, output_dim) * 2 * init_range - init_range
    return nn.Parameter(initial, requires_grad=True)


class GraphConvSparse(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, adj: torch.Tensor, activation=F.leaky_relu) -> None:
        super().__init__()
        self.weight = glorot_init(input_dim, output_dim)
        self.adj = adj
        self.activation = activation

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.mm(x, self.weight)
        x = torch.mm(self.adj, x)
        return self.activation(x)


class GraphTransformerConv(nn.Module):
    """Graph Transformer encoder with Graphormer-style structural bias (covers Graphormer + SAN).
    Full multi-head self-attention; adjacency values inject a per-head learnable edge bias.
    NOTE: O(N^2 * H) memory — suitable for the node sizes in these datasets.
    """
    def __init__(self, input_dim: int, output_dim: int, adj: torch.Tensor,
                 num_heads: int = 4, activation=F.leaky_relu) -> None:
        super().__init__()
        assert output_dim % num_heads == 0, "output_dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim  = output_dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.Q         = nn.Linear(input_dim, output_dim, bias=False)
        self.K         = nn.Linear(input_dim, output_dim, bias=False)
        self.V         = nn.Linear(input_dim, output_dim, bias=False)
        self.out_proj  = nn.Linear(output_dim, output_dim)
        self.edge_bias = nn.Parameter(torch.zeros(num_heads))
        self.adj = adj
        self.activation = activation

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        N = x.size(0)
        Q = self.Q(x).view(N, self.num_heads, self.head_dim)
        K = self.K(x).view(N, self.num_heads, self.head_dim)
        V = self.V(x).view(N, self.num_heads, self.head_dim)
        attn = torch.einsum('ihd,jhd->ijh', Q, K) * self.scale
        struct_bias = self.adj.unsqueeze(-1) * self.edge_bias.view(1, 1, self.num_heads)
        attn = attn + struct_bias
        attn = F.softmax(attn, dim=1)
        out = torch.einsum('ijh,jhd->ihd', attn, V).reshape(N, -1)
        return self.activation(self.out_proj(out))



class GAEMF(nn.Module):
    def __init__(self, adj: torch.Tensor, feature_dim: int, args) -> None:
        super().__init__()
        self.start_mf = args.start_mf
        self.base_gcn = GraphTransformerConv(feature_dim, args.encoded_space_dim, adj)
        self.cluster_centroid = glorot_init(args.n_cluster, args.encoded_space_dim)

    def restart_clusters(self) -> None:
        torch.nn.init.xavier_normal_(self.cluster_centroid.data)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.base_gcn(x)

    @staticmethod
    def normalize(x: torch.Tensor) -> torch.Tensor:
        row_min = x.min(dim=1).values[:, None]
        row_max = x.max(dim=1).values[:, None]
        denom = (row_max - row_min).clamp_min(1e-12)
        x_std = (x - row_min) / denom
        return x_std / x_std.sum(dim=1, keepdim=True).clamp_min(1e-12)

    def forward(self, x: torch.Tensor, flag):
        z = self.encode(x)
        adj_pred = torch.sigmoid(torch.matmul(z, z.t()))
        if type(flag) != bool or flag is True:
            pinv_weight = torch.linalg.pinv(self.cluster_centroid)
            indicator = self.normalize(torch.mm(z, pinv_weight))
            return adj_pred, z, indicator
        return adj_pred, z, None


@dataclass
class ExperimentConfig:
    dataset_name: str = "highschool"
    max_snapshots: int = 10
    encoded_space_dim: int = 64
    n_cluster: Optional[int] = None
    num_pretrain_epoch: int = 250
    num_topo_epoch: int = 150
    start_mf: int = 40
    learning_rate: float = 1e-3
    alpha_cluster: float = 1.0
    lambda_node: float = 0.5
    lambda_community: float = 1.0
    node_topo_max_nodes: int = 96
    node_topo_k: int = 12
    node_topo_temperature: float = 1.0
    topo_cardinality: int = 32
    eval_use_argmax_q: Optional[bool] = None
    unknown_k_multiplier: int = 2
    seed: int = 7
    save_dir: str = "Output/Encoder Variants/multiscale_toporeg_transformer"
    device: Optional[str] = None


@dataclass
class SnapshotData:
    adj_sp: sp.coo_matrix
    adj_dense: torch.Tensor
    adj_norm_dense: torch.Tensor
    features: torch.Tensor
    labels: torch.Tensor


def require_gudhi() -> None:
    if gd is None or wasserstein_distance is None:
        raise ModuleNotFoundError("gudhi is required for persistent homology.")


class DeviceAwareWrcfLayer(torch.nn.Module):
    def __init__(self, dim: int, card: int, device: torch.device) -> None:
        super().__init__()
        self.dim = dim
        self.card = card
        self.device = device

    def forward(self, graph: torch.Tensor) -> torch.Tensor:
        graph_cpu = graph   # KEEP ORIGINAL TENSOR
        # Compute indices WITHOUT breaking graph permanently
        ids_np = wrcf_index(graph.detach().cpu().numpy(), self.dim, self.card)
        ids = torch.from_numpy(ids_np).long().to(graph.device)

        if self.dim > 0:
            indices = ids.view([2 * self.card, 2]).long()
            dgm = graph_cpu[indices[:, 0], indices[:, 1]].view(self.card, 2)
        else:
            indices = ids.view([2 * self.card, 2])[1::2, :].long()
            dgm = torch.cat(
                [
                    torch.zeros(self.card, 1, device=graph_cpu.device),
                    graph_cpu[indices[:, 0], indices[:, 1]].view(self.card, 1).float(),
                ],
                dim=1,
            )
        return dgm.to(self.device)


class MultiScaleTopoLoss(torch.nn.Module):
    def __init__(
        self,
        card: int,
        lambda_node: float,
        lambda_community: float,
        node_target: Tuple[Optional[Tuple[torch.Tensor, torch.Tensor]], Optional[Tuple[torch.Tensor, torch.Tensor]]],
        community_target: Tuple[Optional[Tuple[torch.Tensor, torch.Tensor]], Optional[Tuple[torch.Tensor, torch.Tensor]]],
        node_max_nodes: int,
        node_topk: int,
        node_temperature: float,
        device: torch.device,
    ) -> None:
        super().__init__()
        self.lambda_node = lambda_node
        self.lambda_community = lambda_community
        self.node_target = node_target
        self.community_target = community_target
        self.node_max_nodes = node_max_nodes
        self.node_topk = node_topk
        self.node_temperature = node_temperature
        self.device = device
        self.wrcf_dim0 = DeviceAwareWrcfLayer(dim=0, card=card, device=device)
        self.wrcf_dim1 = DeviceAwareWrcfLayer(dim=1, card=card, device=device)

    def forward(self, adj_dense: torch.Tensor, q: torch.Tensor, z: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, float]]:
        comm_graph = build_soft_community_graph(q, adj_dense)
        comm_dgm = (self.wrcf_dim0(comm_graph), self.wrcf_dim1(comm_graph))

        latent_graph = build_node_topology_graph(
            z=z,
            max_nodes=self.node_max_nodes,
            topk=self.node_topk,
            temperature=self.node_temperature,
        )
        node_dgm = (self.wrcf_dim0(latent_graph), self.wrcf_dim1(latent_graph))

        node_loss = diagram_pair_loss(node_dgm, self.node_target)
        community_loss = diagram_pair_loss(comm_dgm, self.community_target)
        total = self.lambda_node * node_loss + self.lambda_community * community_loss
        stats = {
            "node_topo": float(node_loss.detach().cpu()),
            "community_topo": float(community_loss.detach().cpu()),
        }
        return total, stats


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def resolve_device(device_name: Optional[str]) -> torch.device:
    if device_name:
        return torch.device(device_name)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def resolve_n_cluster(config: ExperimentConfig, label_count: int) -> int:
    if config.n_cluster is not None:
        return config.n_cluster
    if config.dataset_name == "DBLPdyn":
        return max(label_count, label_count * config.unknown_k_multiplier)
    return label_count


def should_use_argmax_q(config: ExperimentConfig) -> bool:
    if config.eval_use_argmax_q is not None:
        return config.eval_use_argmax_q
    return config.dataset_name == "DBLPdyn"


def graph_normalization(adj: sp.coo_matrix) -> sp.coo_matrix:
    adj_ = adj + sp.eye(adj.shape[0])
    rowsum = np.array(adj_.sum(1))
    degree_mat_inv_sqrt = sp.diags(np.power(rowsum, -0.5).flatten())
    return adj_.dot(degree_mat_inv_sqrt).transpose().dot(degree_mat_inv_sqrt).tocoo()


def load_dynamic_dataset(config: ExperimentConfig, device: torch.device) -> Tuple[List[SnapshotData], int]:
    if config.dataset_name not in graph_pkl:
        raise ValueError(f"Unknown dataset: {config.dataset_name}")

    graph_path = Path("Data") / f"{config.dataset_name}.pkl"
    label_path = Path("Data") / f"{config.dataset_name}_label.pkl"
    with graph_path.open("rb") as handle:
        graphs = pickle.load(handle, encoding="bytes", fix_imports=True)
    with label_path.open("rb") as handle:
        labels = pickle.load(handle, encoding="bytes", fix_imports=True)

    if config.max_snapshots:
        graphs = graphs[: config.max_snapshots]
    label_count = label_num_dic[config.dataset_name]

    if config.dataset_name != "DBLPdyn":
        label_set = set(labels.values())
        label_map = {label: idx for idx, label in enumerate(label_set)}
    else:
        label_map = {str(i): i - 1 for i in range(1, label_count + 1)}

    snapshots: List[SnapshotData] = []
    for snapshot_index, graph in enumerate(graphs):
        adj_sp = sp.coo_matrix(nx.adjacency_matrix(graph))
        adj_dense = torch.tensor(adj_sp.toarray(), dtype=torch.float32, device=device)
        adj_norm_sp = graph_normalization(adj_sp)
        adj_norm_dense = torch.tensor(adj_norm_sp.toarray(), dtype=torch.float32, device=device)

        features = torch.eye(adj_sp.shape[0], dtype=torch.float32, device=device)
        if config.dataset_name == "DBLPdyn":
            label_list = [label_map[labels[snapshot_index][node]] for node in graph.nodes()]
        else:
            label_list = [label_map[labels[node]] for node in graph.nodes()]

        snapshots.append(
            SnapshotData(
                adj_sp=adj_sp,
                adj_dense=adj_dense,
                adj_norm_dense=adj_norm_dense,
                features=features,
                labels=torch.tensor(label_list, dtype=torch.long, device=device),
            )
        )

    return snapshots, label_count


def build_soft_community_graph(q: torch.Tensor, adj_dense: torch.Tensor) -> torch.Tensor:
    indicator = torch.argmax(q, dim=1)
    indicator_hat = torch.stack([torch.where(indicator == k, 1.0, 0.0) for k in range(q.size(1))], dim=1)
    q_hat = indicator_hat * q
    comm_graph = q_hat.T @ adj_dense @ q_hat
    comm_graph = comm_graph * (1 - torch.eye(comm_graph.size(0), device=comm_graph.device))
    denom = adj_dense.sum().clamp_min(1.0)
    return comm_graph / denom


def build_node_topology_graph(z: torch.Tensor, max_nodes: int, topk: int, temperature: float) -> torch.Tensor:
    if z.size(0) > max_nodes:
        scores = torch.norm(z, dim=1)
        keep = torch.topk(scores, k=max_nodes).indices
        z = z[keep]

    z = F.normalize(z, p=2, dim=1)
    distances = torch.cdist(z, z, p=2)
    non_zero = distances[distances > 0]
    sigma = non_zero.median() if non_zero.numel() > 0 else torch.tensor(1.0, device=z.device)
    sigma = sigma.clamp_min(1e-6) * temperature
    weights = torch.exp(-torch.square(distances / sigma))
    weights = weights * (1 - torch.eye(weights.size(0), device=weights.device))

    if 0 < topk < weights.size(1):
        knn = torch.topk(weights, k=topk, dim=1).indices
        mask = torch.zeros_like(weights)
        mask.scatter_(1, knn, 1.0)
        mask = torch.maximum(mask, mask.T)
        weights = weights * mask

    denom = weights.max().clamp_min(1.0)
    return weights / denom


def wrcf(graph: np.ndarray) -> gd.SimplexTree:
    require_gudhi()
    nx_graph = nx.from_numpy_array(graph)
    st = gd.SimplexTree()
    for node in nx_graph.nodes():
        st.insert([node], filtration=0.0)

    edge_weights = [weight for _, _, weight in nx_graph.edges.data("weight")]
    distinct_weights = np.unique(edge_weights)[::-1] if edge_weights else []
    for weight in distinct_weights:
        if weight <= 0:
            continue
        subgraph = nx_graph.edge_subgraph([(u, v) for u, v, w in nx_graph.edges.data("weight") if w >= weight])
        for clique in nx.find_cliques(subgraph):
            st.insert(clique, filtration=float(1.0 / max(weight, 1e-12)))
    return st


def wrcf_index(graph: np.ndarray, dim: int, card: int) -> np.ndarray:
    st = wrcf(graph)
    st.persistence()
    pairs = st.persistence_pairs()

    indices: List[int] = []
    persistence_scores: List[float] = []
    for simplex_birth, simplex_death in pairs:
        if len(simplex_birth) != dim + 1 or len(simplex_death) == 0:
            continue
        birth_vertices = np.array(simplex_birth)
        death_vertices = np.array(simplex_death)
        birth_pair = [
            simplex_birth[v]
            for v in np.unravel_index(
                np.argmax(graph[birth_vertices, :][:, birth_vertices]),
                [len(simplex_birth), len(simplex_birth)],
            )
        ]
        death_pair = [
            simplex_death[v]
            for v in np.unravel_index(
                np.argmax(graph[death_vertices, :][:, death_vertices]),
                [len(simplex_death), len(simplex_death)],
            )
        ]
        indices.extend(birth_pair)
        indices.extend(death_pair)
        persistence_scores.append(st.filtration(simplex_death) - st.filtration(simplex_birth))

    if persistence_scores:
        order = np.argsort(persistence_scores)
        indices = list(np.reshape(indices, [-1, 4])[order][::-1, :].flatten())
    indices = indices[: 4 * card] + [0 for _ in range(max(0, 4 * card - len(indices)))]
    return np.array(indices, dtype=np.int32)


def safe_wasserstein(current: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    require_gudhi()
    return wasserstein_distance(
        current.double(),
        target.double(),
        order=1,
        enable_autodiff=True,
        keep_essential_parts=False,
    )


def diagram_pair_loss(
    current: Tuple[torch.Tensor, torch.Tensor],
    target_pair: Tuple[Optional[Tuple[torch.Tensor, torch.Tensor]], Optional[Tuple[torch.Tensor, torch.Tensor]]],
) -> torch.Tensor:
    losses: List[torch.Tensor] = []
    for target in target_pair:
        if target is None:
            continue
        losses.append(safe_wasserstein(current[0], target[0]))
        losses.append(safe_wasserstein(current[1], target[1]))
    if not losses:
        return torch.tensor(0.0, device=current[0].device)
    return torch.stack(losses).sum()


def compute_reconstruction_loss(adj_dense: torch.Tensor, adj_pred: torch.Tensor) -> torch.Tensor:
    positive = adj_dense.sum().clamp_min(1.0)
    negative = torch.tensor(float(adj_dense.numel()), device=adj_dense.device) - positive
    pos_weight = negative / positive
    norm = torch.tensor(float(adj_dense.numel()), device=adj_dense.device) / ((negative * 2.0).clamp_min(1.0))
    weights = torch.ones_like(adj_dense)
    weights = torch.where(adj_dense > 0, pos_weight, weights)
    return norm * F.binary_cross_entropy(adj_pred, adj_dense, weight=weights)


def pretrain_snapshot(model: GAEMF, snapshot: SnapshotData, config: ExperimentConfig) -> List[Dict[str, float]]:
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=1e-4)
    history: List[Dict[str, float]] = []

    model.train()
    for epoch in range(config.num_pretrain_epoch):
        use_mf = epoch >= config.start_mf
        adj_pred, z, q = model(snapshot.features, use_mf)
        recon = compute_reconstruction_loss(snapshot.adj_dense, adj_pred)
        cluster = torch.tensor(0.0, device=snapshot.adj_dense.device)
        if use_mf and q is not None:
            cluster = F.mse_loss(z, q @ model.cluster_centroid)
        loss = recon + config.alpha_cluster * cluster

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(
            {
                "epoch": epoch,
                "loss": float(loss.detach().cpu()),
                "recon": float(recon.detach().cpu()),
                "cluster": float(cluster.detach().cpu()),
            }
        )
    return history


@torch.no_grad()
def extract_snapshot_state(
    model: GAEMF,
    snapshot: SnapshotData,
    config: ExperimentConfig,
    wrcf_dim0: DeviceAwareWrcfLayer,
    wrcf_dim1: DeviceAwareWrcfLayer,
) -> Dict[str, torch.Tensor]:
    model.eval()
    _, z, q = model(snapshot.features, True)
    comm_graph = build_soft_community_graph(q, snapshot.adj_dense)
    node_graph = build_node_topology_graph(
        z=z,
        max_nodes=config.node_topo_max_nodes,
        topk=config.node_topo_k,
        temperature=config.node_topo_temperature,
    )
    return {
        "z": z.detach(),
        "q": q.detach(),
        "comm_dim0": wrcf_dim0(comm_graph),
        "comm_dim1": wrcf_dim1(comm_graph),
        "node_dim0": wrcf_dim0(node_graph),
        "node_dim1": wrcf_dim1(node_graph),
    }


def build_neighbor_targets(states: Sequence[Dict[str, torch.Tensor]], index: int) -> Tuple[
    Tuple[Optional[Tuple[torch.Tensor, torch.Tensor]], Optional[Tuple[torch.Tensor, torch.Tensor]]],
    Tuple[Optional[Tuple[torch.Tensor, torch.Tensor]], Optional[Tuple[torch.Tensor, torch.Tensor]]],
]:
    prev_state = states[index - 1] if index > 0 else None
    next_state = states[index + 1] if index < len(states) - 1 else None

    node_target = (
        (prev_state["node_dim0"], prev_state["node_dim1"]) if prev_state is not None else None,
        (next_state["node_dim0"], next_state["node_dim1"]) if next_state is not None else None,
    )
    community_target = (
        (prev_state["comm_dim0"], prev_state["comm_dim1"]) if prev_state is not None else None,
        (next_state["comm_dim0"], next_state["comm_dim1"]) if next_state is not None else None,
    )
    return node_target, community_target


def finetune_snapshot(model: GAEMF, snapshot: SnapshotData, topo_loss: MultiScaleTopoLoss, config: ExperimentConfig) -> List[Dict[str, float]]:
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=1e-4)
    history: List[Dict[str, float]] = []

    model.train()
    for epoch in range(config.num_topo_epoch):
        adj_pred, z, q = model(snapshot.features, True)
        recon = compute_reconstruction_loss(snapshot.adj_dense, adj_pred)
        cluster = F.mse_loss(z, q @ model.cluster_centroid)
        topo, topo_stats = topo_loss(snapshot.adj_dense, q, z)
        loss = recon + config.alpha_cluster * cluster + topo

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(
            {
                "epoch": epoch,
                "loss": float(loss.detach().cpu()),
                "recon": float(recon.detach().cpu()),
                "cluster": float(cluster.detach().cpu()),
                "node_topo": topo_stats["node_topo"],
                "community_topo": topo_stats["community_topo"],
            }
        )
    return history


@torch.no_grad()
def evaluate_snapshot(model: GAEMF, snapshot: SnapshotData, config: ExperimentConfig) -> Dict[str, float]:
    model.eval()
    adj_pred, z, q = model(snapshot.features, True)
    use_argmax_q = should_use_argmax_q(config)
    predicted = (
        torch.argmax(q, dim=1).cpu().numpy()
        if use_argmax_q
        else KMeans(n_clusters=config.n_cluster, n_init=20).fit_predict(z.cpu().numpy())
    )
    labels = snapshot.labels.cpu().numpy()

    acc = clustering_accuracy(labels, predicted)
    nmi = metrics.normalized_mutual_info_score(labels, predicted)
    ari = metrics.adjusted_rand_score(labels, predicted)
    mod = modularity(snapshot.adj_sp, predicted)
    recon_acc = float((adj_pred > 0.5).eq(snapshot.adj_dense > 0).float().mean().cpu())
    return {"acc": acc, "nmi": float(nmi), "ari": float(ari), "modularity": float(mod), "recon_acc": recon_acc}


def clustering_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.array(y_true, dtype=np.int64).copy()
    y_pred = np.array(y_pred, dtype=np.int64)
    voted = np.zeros_like(y_true)
    labels = np.unique(y_true)
    ordered = np.arange(labels.shape[0])
    for idx, label in enumerate(labels):
        y_true[y_true == label] = ordered[idx]
    bins = np.concatenate((np.unique(y_true), [np.max(y_true) + 1]), axis=0)
    for cluster in np.unique(y_pred):
        hist, _ = np.histogram(y_true[y_pred == cluster], bins=bins)
        winner = np.argmax(hist) if hist.size else 0
        voted[y_pred == cluster] = winner
    return float(metrics.accuracy_score(y_true, voted))


def modularity(adjacency_matrix: sp.coo_matrix, label_list: np.ndarray) -> float:
    labels = np.asarray(label_list, dtype=np.int64)
    num_classes = int(labels.max()) + 1 if labels.size else 0
    indicator = np.eye(num_classes)[labels] if num_classes else np.zeros((0, 0))
    adjacency = adjacency_matrix.toarray()
    m = np.sum(adjacency) / 2
    if m == 0:
        return 0.0
    degree = np.sum(adjacency, axis=1)
    modularity_matrix = adjacency - np.outer(degree, degree) / (2 * m)
    return float((1 / (2 * m)) * np.trace(indicator.T @ modularity_matrix @ indicator))


def train_multiscale_toporeg(config):

    require_gudhi()
    set_seed(config.seed)
    device = resolve_device(config.device)

    snapshots, label_count = load_dynamic_dataset(config, device)
    config.n_cluster = resolve_n_cluster(config, label_count)

    models = [
      GAEMF(s.adj_norm_dense, s.features.size(1), config).to(device)
      for s in snapshots
    ]

    optimizers = [
      torch.optim.AdamW(m.parameters(), lr=config.learning_rate)
      for m in models
    ]

    # TRAINING LOOP
    for epoch in range(config.num_topo_epoch):

        # forward all snapshots
        zs, qs, adjs = [], [], []
        for model, snapshot in zip(models, snapshots):
            adj_pred, z, q = model(snapshot.features, True)
            zs.append(z)
            qs.append(q)
            adjs.append(snapshot.adj_dense)

        total_loss = 0

        # compute losses
        for t in range(len(models)):
          # ---- Reconstruction loss ----
          recon = compute_reconstruction_loss(
            adjs[t],
            torch.sigmoid(zs[t] @ zs[t].T)
          )

          # ---- Clustering loss ----
          cluster = F.mse_loss(
            zs[t],
            qs[t] @ models[t].cluster_centroid
          )

          # ---- Topological loss ----
          topo_loss_val = 0

          if t > 0:
            topo_loss_val += compute_topo_pair_loss(
              zs[t], qs[t], adjs[t],
              zs[t-1], qs[t-1], adjs[t-1],
              config
            )

          if t < len(models) - 1:
            topo_loss_val += compute_topo_pair_loss(
              zs[t], qs[t], adjs[t],
              zs[t+1], qs[t+1], adjs[t+1],
              config
              )

          # ---- Total loss ----
          loss = recon + config.alpha_cluster * cluster + topo_loss_val
          total_loss += loss

        # backward
        for opt in optimizers:
            opt.zero_grad()

        total_loss.backward()

        for opt in optimizers:
            opt.step()


    metrics_by_snapshot = []

    for model, snapshot in zip(models, snapshots):
        metrics = evaluate_snapshot(model, snapshot, config)
        metrics_by_snapshot.append(metrics)

    summary = {
        "config": asdict(config),
        "device": str(device),
        "metrics_by_snapshot": metrics_by_snapshot,
        "mean_metrics": {
            key: float(np.mean([m[key] for m in metrics_by_snapshot]))
            for key in metrics_by_snapshot[0].keys()
        },
    }

    return summary

def compute_topo_pair_loss(z1, q1, adj1, z2, q2, adj2, config):

    # ---- Community graphs ----
    comm1 = build_soft_community_graph(q1, adj1)
    comm2 = build_soft_community_graph(q2, adj2)

    # ---- Node graphs ----
    node1 = build_node_topology_graph(
        z1,
        max_nodes=config.node_topo_max_nodes,
        topk=config.node_topo_k,
        temperature=config.node_topo_temperature,
    )

    node2 = build_node_topology_graph(
        z2,
        max_nodes=config.node_topo_max_nodes,
        topk=config.node_topo_k,
        temperature=config.node_topo_temperature,
    )

    # ---- Diagrams ----
    wrcf0 = DeviceAwareWrcfLayer(0, config.topo_cardinality, z1.device)
    wrcf1 = DeviceAwareWrcfLayer(1, config.topo_cardinality, z1.device)

    comm_dgm1 = (wrcf0(comm1), wrcf1(comm1))
    comm_dgm2 = (wrcf0(comm2), wrcf1(comm2))

    node_dgm1 = (wrcf0(node1), wrcf1(node1))
    node_dgm2 = (wrcf0(node2), wrcf1(node2))

    # ---- Loss ----
    comm_loss = diagram_pair_loss(comm_dgm1, (comm_dgm2,))
    node_loss = diagram_pair_loss(node_dgm1, (node_dgm2,))

    return (
        config.lambda_community * comm_loss +
        config.lambda_node * node_loss
    )

def format_summary(summary: Dict[str, object]) -> str:
    metrics_block = summary["mean_metrics"]
    lines = [
        f"Device: {summary['device']}",
        f"ACC: {metrics_block['acc']:.4f}",
        f"NMI: {metrics_block['nmi']:.4f}",
        f"ARI: {metrics_block['ari']:.4f}",
        f"Modularity: {metrics_block['modularity']:.4f}",
        f"Reconstruction ACC: {metrics_block['recon_acc']:.4f}",
    ]
    return "\n".join(lines)


1.12.1+cu113
True
Tesla K80


In [3]:
%cd  /home/others/CS60078_P7/CNTGroup7project

/home/others/CS60078_P7/CNTGroup7project


/home/others/CS60078_P7/miniconda3/envs/torch_env/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [4]:
config = ExperimentConfig()
summary = train_multiscale_toporeg(config)
print(format_summary(summary))

Device: cuda
ACC: 0.2397
NMI: 0.0938
ARI: 0.0135
Modularity: 0.0054
Reconstruction ACC: 0.0175
